In [17]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, OneHotEncoder
import pickle
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, TensorBoard
import datetime

# ---------------------------
# Data Loading and Preprocessing
# ---------------------------
# Load the dataset
data = pd.read_csv("Churn_Modelling.csv")
print("Original data:")
print(data.head())

# Drop irrelevant columns
data = data.drop(['RowNumber', 'CustomerId', 'Surname'], axis=1)
print("\nAfter dropping unnecessary columns:")
print(data.head())

# Encode the categorical variable 'Gender'
label_encoder_gender = LabelEncoder()
data['Gender'] = label_encoder_gender.fit_transform(data['Gender'])
print("\nAfter encoding 'Gender':")
print(data.head())

# One-hot encode the 'Geography' column
onehot_encoder_geo = OneHotEncoder()
geo_encoded = onehot_encoder_geo.fit_transform(data[['Geography']]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
print("\nOne-hot encoded 'Geography':")
print(geo_encoded_df.head())

# Combine one-hot encoded 'Geography' with the original data (dropping original 'Geography')
data = pd.concat([data.drop('Geography', axis=1), geo_encoded_df], axis=1)
print("\nData after combining one-hot encoded Geography:")
print(data.head())

# Save the encoders for future use
with open('label_encoder_gender.pkl', 'wb') as file:
    pickle.dump(label_encoder_gender, file)

with open('onehot_encoder_geo.pkl', 'wb') as file:
    pickle.dump(onehot_encoder_geo, file)

# ---------------------------
# Splitting Data and Scaling Features
# ---------------------------
# Divide the dataset into independent (X) and dependent (y) features
X = data.drop('Exited', axis=1)
y = data['Exited']

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale the features
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Save the scaler for future predictions
with open('scaler.pkl', 'wb') as file:
    pickle.dump(scaler, file)

# ---------------------------
# Define and Train the ANN Model
# ---------------------------
def create_improved_model(neurons=32, layers=2, dropout_rate=0.2):
    """
    Builds an improved ANN model with batch normalization and dropout.
    """
    model = Sequential()
    # First hidden layer with Batch Normalization and Dropout
    model.add(Dense(neurons, activation='relu', input_shape=(X_train.shape[1],)))
    model.add(BatchNormalization())
    model.add(Dropout(dropout_rate))
    
    # Add additional hidden layers with Batch Normalization and Dropout
    for _ in range(layers - 1):
        model.add(Dense(neurons, activation='relu'))
        model.add(BatchNormalization())
        model.add(Dropout(dropout_rate))
        
    # Output layer
    model.add(Dense(1, activation='sigmoid'))
    return model

# Create the model instance
model = create_improved_model()
model.summary()  # Display the model architecture

# Define optimizer and loss function
opt = tf.keras.optimizers.Adam(learning_rate=0.01)
loss_function = tf.keras.losses.BinaryCrossentropy()

# Compile the model
model.compile(optimizer=opt, loss=loss_function, metrics=['accuracy'])

# ---------------------------
# Set Up Callbacks and Train the Model
# ---------------------------
# Set up TensorBoard logging (ensure that "logs/fit" exists or will be created)
log_dir = "logs/fit/" + datetime.datetime.now().strftime("%Y%m%d-%H%M%S")
tensorboard_callback = TensorBoard(log_dir=log_dir, histogram_freq=1)

# Set up Early Stopping to prevent overfitting
early_stopping_callback = EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# Train the model
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    callbacks=[tensorboard_callback, early_stopping_callback]
)

# Save the trained model
model.save('model.h5')


Original data:
   RowNumber  CustomerId   Surname  CreditScore Geography  Gender  Age  \
0          1    15634602  Hargrave          619    France  Female   42   
1          2    15647311      Hill          608     Spain  Female   41   
2          3    15619304      Onio          502    France  Female   42   
3          4    15701354      Boni          699    France  Female   39   
4          5    15737888  Mitchell          850     Spain  Female   43   

   Tenure    Balance  NumOfProducts  HasCrCard  IsActiveMember  \
0       2       0.00              1          1               1   
1       1   83807.86              1          0               1   
2       8  159660.80              3          1               0   
3       1       0.00              2          0               0   
4       2  125510.82              1          1               1   

   EstimatedSalary  Exited  
0        101348.88       1  
1        112542.58       0  
2        113931.57       1  
3         93826.63       0 

d:\annclassification\env\lib\site-packages\keras\src\engine\training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(
